email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [3]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path to import azure_openai_config
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'))

load_dotenv()

from azure_openai_config import setup_azure_openai, get_azure_openai_model
setup_azure_openai()


In [4]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [5]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [6]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

In [8]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = """You are a helpful assistant that can authenticate users. 
Before accessing the inbox, users must authenticate. Ask them for their email address and password, 
then use the authenticate tool to verify their credentials."""

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model=get_azure_openai_model(),
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Please check my inbox")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

To access your inbox, I need to authenticate you first. Could you please provide your email address and password?


In [12]:
# Continue the conversation by providing credentials
response = agent.invoke(
    {"messages": [HumanMessage(content="My email is julie@example.com and password is password123")]},
    context=EmailContext(),
    config=config  # Same thread_id to continue the conversation
)

print(response['messages'][-1].content)

c:\Users\manikanta.reddy\source\repos\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


You have an email from Jane (jane@example.com). She mentioned that she'll be in town next week and is wondering if you could grab a coffee together. Would you like to respond to her?


In [14]:
# Check if there's an interrupt (email waiting for approval)
if '__interrupt__' in response:
    print("Email body waiting for approval:")
    print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])
else:
    print("No interrupt - agent may be waiting for more input or already completed.")
    print("\nLast message:")
    print(response['messages'][-1].content)

No interrupt - agent may be waiting for more input or already completed.

Last message:
You have an email from Jane (jane@example.com). She mentioned that she'll be in town next week and is wondering if you could grab a coffee together. Would you like to respond to her?


In [15]:
# Ask the agent to send a reply
response = agent.invoke(
    {"messages": [HumanMessage(content="Please send a reply to Jane accepting the coffee invitation")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content if '__interrupt__' not in response else "Waiting for approval...")

Waiting for approval...


In [16]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Your reply has been sent to Jane, accepting her coffee invitation. Let me know if there's anything else you'd like to do!


In [17]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='Please check my inbox', additional_kwargs={}, response_metadata={}, id='2c3730ef-0441-4f4e-bf0d-160d0e1f1227'),
              AIMessage(content='To access your inbox, I need to authenticate you first. Could you please provide your email address and password?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 94, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_b54fe76834', 'id': 'chatcmpl-CrGrLtbOt9XwurVVSJ444ay5tbE12', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_har